# 可访问性与 HTML 检查

学习目标：能检查页面名称、语言和键盘操作，修正常见 HTML 问题，并说明自动检查的局限。

前置知识：HTML 文档结构、语义元素、链接、表单控件和浏览器开发者工具的基本操作。

适用范围：WHATWG HTML Living Standard；使用现代浏览器和键盘。开发者工具步骤以 Chrome 为例，屏幕阅读器朗读需在实际使用的浏览器与辅助技术组合中另测。

环境准备：[环境配置与运行](README.md)。

工作目录：content/Web与应用开发/html；在浏览器运行配套页面。

配套脚本：位于 scripts/13-accessibility-and-html-checks/。

1. [broken.html](scripts/13-accessibility-and-html-checks/broken.html)、[fixed.html](scripts/13-accessibility-and-html-checks/fixed.html)：待修正页与修正页，对照标签关联、原生按钮和文档结构。
2. [focus.html](scripts/13-accessibility-and-html-checks/focus.html)、[focus.js](scripts/13-accessibility-and-html-checks/focus.js)：tabindex、程序聚焦与视觉顺序实验。
3. [inspect.js](scripts/13-accessibility-and-html-checks/inspect.js)：两页共用的“标为已读”动作，不承担 HTML 校验或可访问名称计算。
4. [styles.css](scripts/13-accessibility-and-html-checks/styles.css)：阅读样式、可见焦点及视觉重排反例。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/html
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8013 --bind 127.0.0.1
```

Step 3：打开[修正示例](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)。

服务根目录为 content/Web与应用开发/html

Notebook 中的文件链接相对于 content/Web与应用开发/html；页面中的 styles.css 等相对 URL 则从页面所在目录解析。两页输入仅用于本地操作，不保存、不提交。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 可访问名称：说明控件用途

可访问名称（accessible name）是辅助技术用来识别元素用途的文字。输入框优先用可见的 &lt;label&gt;，按钮优先用按钮内的文字。

- for：&lt;label&gt; 的属性，值要对应目标控件的唯一 id。
- id：标识文档中的元素；这里的 book、reader 是自定标识符。
- name：表单提交字段名，与可访问名称是不同概念。
- placeholder：输入提示；输入后会消失，不能替代始终可见的标签。

```html
<p><label for="book">书名</label><input id="book" name="book" placeholder="例如：小王子"></p>
<p><label for="reader">记录人</label><input id="reader" name="reader"></p>
<!-- 分别点击“书名”“记录人”，焦点应进入对应输入框；输入后标签仍然可见。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

### 1.1 必要时补充 ARIA 名称

ARIA 用属性补充辅助技术需要的语义。本例用 aria-label 为导航区域命名：

- aria-label：直接提供名称文字，不会在页面上新增可见文字。
- aria-labelledby：引用已有文字所在元素的 id；引用多个 id 时用空格分隔。

命名方式须适用于对应角色。已有清楚标签的控件通常不需要再加 ARIA 名称，也不要给每段普通文字都命名。

```html
<nav aria-label="示例页面">
  <a href="broken.html">待修正示例</a>
  <a href="focus.html">焦点顺序实验</a>
</nav>
<!-- 在开发者工具的 Accessibility 中检查导航区域名称“示例页面”。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

## 2 lang：声明内容语言

lang 是全局属性。通常在 &lt;html&gt; 的开始标签上声明页面主要语言；局部内容需要时另行声明。普通子元素未声明时沿用祖先提供的语言。

- zh-CN：本例的中文语言标签，含中国地区信息。
- en：英语。

lang 帮助浏览器和辅助技术确定内容语言，不会翻译文字。修正页的文档起始部分为：

```html
<!doctype html>
<html lang="zh-CN">
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

同一文件的主要内容中有一段英语：

```html
<p lang="en">Read, think, and write.</p>
<!-- 检查页面与英语段落的 lang；是否切换朗读语言还需用目标屏幕阅读器确认。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

## 3 优先使用原生按钮

“标为已读”是一个动作，使用 &lt;button&gt;。type="button" 表示普通按钮；按钮文字提供名称，原生按钮支持键盘激活。

role="status" 将下方段落标为非紧急状态提示区域。脚本只更新文字，不将焦点移到提示；具体朗读效果仍需实测。

```html
<button type="button" data-action="mark">标为已读</button>
<p id="action-result" role="status">尚未标记。</p>
<!-- 按 Tab 聚焦按钮，分别试 Enter 和空格；每次测试前刷新，应能把提示改为“已标为已读”。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

### 3.1 能聚焦，不等于能用键盘激活

反例给 &lt;div&gt; 加了按钮角色和 tabindex。它能进入 Tab 顺序，但 ARIA 不会自动补齐 Enter、空格的处理，也不会生成按钮样式。

两页使用相同的 click 处理器。下面只改变承载动作的元素，用来比较原生行为：

```html
<div class="pretend-button" role="button" tabindex="0" data-action="mark">标为已读</div>
<p id="action-result" role="status">尚未标记。</p>
<!-- 普通键盘操作下，Enter 和空格不会触发本例的 div 动作；鼠标点击可以。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/broken.html](scripts/13-accessibility-and-html-checks/broken.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/broken.html)

inspect.js 使用浏览器 DOM API 监听动作并更新文字，无需在 Notebook 中执行：

```javascript
document.querySelector("[data-action=mark]").addEventListener("click", () => {
  document.querySelector("#action-result").textContent = "已标为已读。";
});
```

配套文件：[scripts/13-accessibility-and-html-checks/inspect.js](scripts/13-accessibility-and-html-checks/inspect.js) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/broken.html)

## 4 阅读顺序与焦点顺序

阅读顺序包含正文；Tab 主要在参与顺序导航的元素之间移动，不会逐段朗读内容。焦点顺序应保留内容含义和操作关系，不要求把所有静态文字都加入 Tab 顺序。

修正页把“跳到主要内容”作为 &lt;body&gt; 内的第一个链接：

```html
<a href="#main">跳到主要内容</a>
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

站点页头和导航之后，主要内容从以下开始标签进入。片段 #main 对应 id="main"；tabindex="-1" 使该区域可以取得焦点，而不增加普通 Tab 停靠点。

```html
<main id="main" tabindex="-1">
  <h1>读书记录：修正示例</h1>
<!-- 从页首用键盘激活跳转链接，检查焦点进入 main；再按 Tab 应进入“书名”输入框。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

### 4.1 tabindex 的 0 与 -1

下面观察普通、可显示且未禁用、未设为 inert 的元素；浏览器和系统键盘设置也会影响导航。

- 0：加入顺序焦点导航；本例按文档顺序到达。
- -1：通常不在 Tab 顺序中，仍可用 focus() 按需聚焦。

focus() 是浏览器 API，只改变焦点，不移动文档节点，也不把 -1 改为 0。

```html
<p tabindex="0" id="zero">0：参与顺序导航的观察段落。</p>
<p tabindex="-1" id="negative">-1：Tab会跳过；点击上方按钮可以把焦点移到这里。</p>
<!-- 用 Tab 对照两段：经过 0，跳过 -1；“把焦点移到说明段”按钮可聚焦 -1。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/focus.html](scripts/13-accessibility-and-html-checks/focus.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/focus.html)

页面上的按钮通过以下必要脚本聚焦说明段；其余脚本只显示当前焦点：

```javascript
document.querySelector("#move-focus").addEventListener("click", () => {
  document.querySelector("#negative").focus(); // 焦点进入说明段，底部焦点名变为 negative。
});
```

配套文件：[scripts/13-accessibility-and-html-checks/focus.js](scripts/13-accessibility-and-html-checks/focus.js) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/focus.html)

### 4.2 正整数会改变优先顺序

在本例同一导航范围内，正数 tabindex 排在普通控件和 0 之前；较小正数在前，相同正数按文档顺序排列。优先调整 HTML 顺序，避免用正数修补布局。

下面故意让数值顺序与源码顺序相反：

```html
<button type="button" tabindex="2">A：源码在前，数值为2</button>
<button type="button" tabindex="1">B：源码在后，数值为1</button>
<!-- 重新加载后从地址栏按 Tab 进入页面，先到 B，再到 A，之后才到普通控件。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/focus.html](scripts/13-accessibility-and-html-checks/focus.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/focus.html)

### 4.3 视觉重排不等于焦点重排

本例仅用 CSS 的 flex-direction: row-reverse 反转三个按钮的视觉位置，未设置 reading-flow、正数 tabindex，也未用脚本重排节点。Tab 仍沿用源码中的三步顺序。

阅读顺序、视觉顺序和焦点顺序不必逐项相同，但不能因此让用户误解步骤或难以完成操作。

```html
<div class="reverse">
  <button type="button">第一步：选书</button>
  <button type="button">第二步：记录</button>
  <button type="button">第三步：复习</button>
</div>
<!-- 在能横排三个按钮的窗口中，从左到右看到“复习、记录、选书”；Tab 则按“选书、记录、复习”移动。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/focus.html](scripts/13-accessibility-and-html-checks/focus.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/focus.html)

reverse 是本例的 class 名称，对应以下辅助样式：

```css
.reverse { display: flex; flex-direction: row-reverse; justify-content: flex-end; flex-wrap: wrap; }
```

配套文件：[scripts/13-accessibility-and-html-checks/styles.css](scripts/13-accessibility-and-html-checks/styles.css) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/focus.html)

## 5 保留可见焦点

键盘用户需要辨认当前焦点。此处用 CSS 的 :focus 选择已聚焦元素，并绘制轮廓；不要移除焦点提示。样式只在配套页面生效。

按 Tab、Shift+Tab 检查前后移动；链接用 Enter 打开，按钮分别用 Enter、空格测试。

```css
:focus { outline: 3px solid #175cd3; outline-offset: 3px; }
```

配套文件：[scripts/13-accessibility-and-html-checks/styles.css](scripts/13-accessibility-and-html-checks/styles.css) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

## 6 检查重复 id 与错误嵌套

### 6.1 重复 id 会破坏标签关联

同一文档中的 id 值必须唯一。反例把两个输入框都写成 entry，两个 &lt;label&gt; 的 for 因而匹配到同一个控件。

```html
<p><label for="entry">书名</label><input id="entry" name="book"></p>
<p><label for="entry">记录人</label><input id="entry" name="reader"></p>
<!-- 点击“记录人”却聚焦第一个输入框；在 Accessibility 中对照两框的计算名称。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/broken.html](scripts/13-accessibility-and-html-checks/broken.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/broken.html)

修正方法见第 1 节：分别使用 book、reader，并同步修改对应的 for。

### 6.2 浏览器能显示，不代表嵌套正确

&lt;p&gt; 的内容模型允许短语内容（phrasing content），例如文字和 &lt;span&gt;，不允许把 &lt;div&gt; 嵌在其中。下面是故意的无效源码：

```html
<p id="intro">阅读说明<div id="inner-block">每天记录一个问题。</div>保留自己的想法。</p>
<!-- 对照源代码与 Elements：inner-block 不会成为 intro 段落的子元素。 -->
```

配套文件：[scripts/13-accessibility-and-html-checks/broken.html](scripts/13-accessibility-and-html-checks/broken.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/broken.html)

浏览器会按 HTML 解析规则恢复错误。修正页把这段说明写成一个普通段落：

```html
<div id="nesting-case">
  <p id="intro">阅读说明：每天记录一个问题，保留自己的想法。</p>
</div>
```

配套文件：[scripts/13-accessibility-and-html-checks/fixed.html](scripts/13-accessibility-and-html-checks/fixed.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/fixed.html)

## 7 HTML 校验与资源加载检查

用篇末 Nu Html Checker 的文件上传或文本输入检查完整的 broken.html、fixed.html，按提示定位重复 id、嵌套和属性问题。不要只上传脱离文档上下文的片段。在线检查器不能直接读取你电脑上的 127.0.0.1 页面。

校验结果可能随工具规则更新；零错误不是符合性认证，更不能证明页面对所有人可用。Python 的 HTMLParser 也不是完整 HTML 校验器，不会检查所有开始、结束标签是否匹配。

资源加载要另看 Network（网络）面板。打开开发者工具后刷新反例页，核对请求 URL 与响应状态：

```html
<link rel="stylesheet" href="styles.css">
<!-- 此文件故意不存在；在 Network 中观察404。 -->
<link rel="stylesheet" href="missing.css">
```

配套文件：[scripts/13-accessibility-and-html-checks/broken.html](scripts/13-accessibility-and-html-checks/broken.html) · [浏览器预览](http://127.0.0.1:8013/scripts/13-accessibility-and-html-checks/broken.html)

missing.css 是故意缺失的资源，不要创建占位文件来消除报错。这里的完整请求路径是 /scripts/13-accessibility-and-html-checks/missing.css；修正页不再引用它。HTTP 200 只说明该次请求成功，不证明 HTML 内容正确。

## 8 结合工具与人工检查

- HTML 检查器：发现可检测的语法、嵌套和属性问题，不判断所有实际使用体验。
- 浏览器：在 Elements 中检查实际 DOM，在 Accessibility 中检查计算名称和角色，在 Network 中检查资源；这些信息不等于屏幕阅读器的实际朗读。
- 键盘与辅助技术：检查名称是否清楚、操作是否可达、焦点顺序是否合理，以及动作后能否理解反馈。

先从页面标题、语言、标题层级、图片替代文本、控件标签和键盘操作检查起，再结合内容检查对比度、放大阅读及动态状态反馈。自动报告与这组初步检查都不能单独证明完整可访问性。

## 本章小结

- 可访问名称说明用途；id 负责关联，name 负责提交字段，三者不能混淆。
- 原生元素提供基本语义和行为；ARIA 不自动实现键盘操作。
- 先安排合理的文档顺序，再检查焦点；能聚焦与能操作是不同问题。
- HTML 校验、资源加载、键盘和辅助技术检查各有范围，需要结合使用。

## 练习

（1）将 scripts/13-accessibility-and-html-checks/broken.html 复制为 scripts/13-accessibility-and-html-checks/practice-broken.html，修正两个输入框的标签关联，并把动作控件改成原生按钮。检查：点击标签分别聚焦对应控件；Tab 可到按钮，Enter 和空格都能更新状态。

（2）将 scripts/13-accessibility-and-html-checks/focus.html 复制为 scripts/13-accessibility-and-html-checks/practice-focus.html，去掉反例中的正数 tabindex，并让三步按钮的视觉顺序与 Tab 顺序都为“选书 → 记录 → 复习”。检查：不增加正数 tabindex，-1 段落仍可由按钮聚焦。

（3）分别用 Nu Html Checker 和纯键盘检查 fixed.html，分开记录工具提示与操作结果。至少检查标签、跳过导航和按钮反馈，并列出一项仍需屏幕阅读器验证的内容。

### 提示

第一题同步修改 for 与 id；第二题检查 reverse 类带来的视觉重排。副本沿用同目录的脚本和样式；练习后可删除副本。

### 参考解析

（1）两个输入框分别使用 book、reader，标签的 for 同步对应。动作改为 type="button" 的 button，保留 data-action="mark"，即可沿用现有 click 监听。只增加 role 或 tabindex 不会实现原生键盘激活。题目只要求修改这两处；原副本的缺少 lang、错误嵌套与 missing.css 仍需分别处理，不能据此宣布整页检查通过。

（2）删除两个正数 tabindex 后，普通按钮按文档顺序参与导航。去掉三步容器的 reverse 类（或将副本样式改为正常行方向），即可让三步的视觉与 DOM 顺序一致；不要再用正数 tabindex 对冲视觉顺序。保留 negative 的 -1 与原脚本，点击聚焦按钮仍能到达该段。

（3）HTML 检查器结果单独记录规则提示，纯键盘结果记录实际经过的控件和操作后的变化。例如跳转进入 main、下一站为书名、按钮能更新提示，属于操作结果；状态提示是否被屏幕阅读器及时朗读、英语段落是否切换发音，仍需要对应辅助技术组合实测。没有提示不能推出所有使用方式都可访问。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| WHATWG HTML | [Global attributes](https://html.spec.whatwg.org/multipage/dom.html#global-attributes)、[lang](https://html.spec.whatwg.org/multipage/dom.html#attr-lang)、[&lt;label&gt;](https://html.spec.whatwg.org/multipage/forms.html#the-label-element)：id 唯一性、语言声明及标签关联；[&lt;p&gt;](https://html.spec.whatwg.org/multipage/grouping-content.html#the-p-element)的内容模型及 [in body 解析规则](https://html.spec.whatwg.org/multipage/parsing.html#parsing-main-inbody)的错误恢复；[tabindex](https://html.spec.whatwg.org/multipage/interaction.html#attr-tabindex)的取值、导航顺序及用户代理条件。 |
| W3C WAI | [Labeling Controls](https://www.w3.org/WAI/tutorials/forms/labels/)的可见标签与 [Names and Descriptions](https://www.w3.org/WAI/ARIA/apg/practices/names-and-descriptions/)的 ARIA 命名条件；[Read Me First](https://www.w3.org/WAI/ARIA/apg/practices/read-me-first/)的角色与行为边界；[Button Pattern](https://www.w3.org/WAI/ARIA/apg/patterns/button/)的键盘激活；[Focus Order](https://www.w3.org/WAI/WCAG22/Understanding/focus-order.html)的含义、阅读与焦点顺序；[G1](https://www.w3.org/WAI/WCAG22/Techniques/general/G1#tests)的跳过导航检查；[WAI-ARIA 1.2 status](https://www.w3.org/TR/wai-aria-1.2/#status)的非紧急状态区域；[Easy Checks](https://www.w3.org/WAI/test-evaluate/easy-checks/)的初步检查范围。 |
| MDN | [Accessible name](https://developer.mozilla.org/en-US/docs/Glossary/Accessible_name)、[placeholder](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Attributes/placeholder)、[tabindex](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Global_attributes/tabindex)、[HTMLElement.focus()](https://developer.mozilla.org/en-US/docs/Web/API/HTMLElement/focus)、[status role](https://developer.mozilla.org/en-US/docs/Web/Accessibility/ARIA/Reference/Roles/status_role)、[:focus](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:focus)和 [Ordering flex items](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Flexible_box_layout/Ordering_items)：名称、提示、程序聚焦与视觉顺序边界。配套脚本使用 [addEventListener()](https://developer.mozilla.org/en-US/docs/Web/API/EventTarget/addEventListener)、[textContent](https://developer.mozilla.org/en-US/docs/Web/API/Node/textContent)和 [focusin](https://developer.mozilla.org/en-US/docs/Web/API/Element/focusin_event)。 |
| Chrome for Developers | [Accessibility reference](https://developer.chrome.com/docs/devtools/accessibility/reference/#tab)的可访问性树和计算属性；[Network reference](https://developer.chrome.com/docs/devtools/network/reference)的请求 URL、状态及刷新检查。 |
| Nu Html Checker | [检查入口](https://validator.w3.org/nu/)与 [About](https://validator.w3.org/nu/about.html)：完整文档检查、规则变化与非认证性质。 |
| Python 3.12 | [http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface)的服务目录、端口和绑定地址；[HTMLParser](https://docs.python.org/3.12/library/html.parser.html#html.parser.HTMLParser)的能力边界。 |